# Notebook 04 — Reliability-Aware Agentic RAG

## Purpose

This notebook demonstrates the reliability mechanisms built on top of the baseline multi-agent RAG system.

**Scope of this notebook (Mechanisms A, E, G):**
- **[A] EvidenceSufficiencyChecker** — estimates whether retrieved evidence is sufficient before generating an answer
- **[G] RecoveryAgent** — adaptive mechanism that changes retrieval strategy or rewrites the query on failure
- **[E] AbstentionGate** — unified terminal failure handler; issues a structured abstention response

**Teammate placeholders:**
- [B] GroundednessVerifier — to be implemented by Robin
- [H] TrustScorer — to be implemented by Daniel

## Pipeline (Alin's components highlighted)

```
Query
  └─► RetrievalCoordinator  (BM25 + Dense + GraphRAG)
           └─► [A] EvidenceAnalyst
                    ├── sufficient  ──────────────────────► AnswerSynthesizer
                    │                                             │
                    │                                       [B] GroundednessVerifier  ← teammate
                    │                                             │
                    │                                       [H] TrustScorer           ← teammate
                    │                                             ├── trust OK ──► Final Response
                    │                                             └── trust low ─┐
                    │                                                             │
                    ├── insufficient ──► [G] RecoveryAgent                       │
                    │                         ├── retry ──► RetrievalCoordinator │
                    │                         └── exhausted ──┐                  │
                    │                                          │                  │
                    └── hard floor ────────────────────────────▼──────────────────▼
                                                         [E] AbstentionGate
```

## Prerequisites
- Completed Notebook 02 (baseline retrievers built and stored in Google Drive)

## Section 1 — Installation

In [ ]:
!pip install --quiet --upgrade pip setuptools wheel
!pip install "git+https://github.com/dhub100/advanced-genai-rag.git@main#subdirectory=baseline/package/"

In [ ]:
import json
import time
import pathlib
import os

import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Section 2 — Configuration

In [ ]:
ROOT = pathlib.Path("/content/drive/MyDrive/Adv_GenAI_FS26")

PATH_BM25_PICKLE  = ROOT / "storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl"
PATH_DENSE_LOADER = ROOT / "storage/subsample/vectordb_dense/load_dense_fixed.py"
PATH_GRAG_LOADER  = ROOT / "storage/subsample/retrieval_graph/load_graphrag.py"
PATH_QA           = ROOT / "advanced-genai-rag/baseline/benchmark/benchmark_qa_bilingual.json"
PATH_QRELS        = ROOT / "advanced-genai-rag/baseline/benchmark/score/fixed_size"

for name, p in [("BM25", PATH_BM25_PICKLE), ("Dense", PATH_DENSE_LOADER),
                ("GraphRAG", PATH_GRAG_LOADER), ("QA", PATH_QA)]:
    status = "✓" if pathlib.Path(p).exists() else "✗ NOT FOUND"
    print(f"  {status}  {name}: {p}")

## Section 3 — Load Baseline System

Load the pre-built retrievers from Notebook 02.  
Refer to Notebook 02 for detailed explanations of each component.

In [ ]:
from rag.retrieval.agents.bm25 import QEBM25, load_bm25_fixed_qe
from rag.retrieval.agents.dense import DenseRetriever, load_dense_fixed
from rag.retrieval.agents.graph import GraphAgent, load_graph_rag
from rag.retrieval.agents.answer_synthesizer import AnswerSynthesizerAgent
from rag.retrieval.orchestrator import Orchestrator

bm25_fixed_qe = load_bm25_fixed_qe(bm25_pickle_path=str(PATH_BM25_PICKLE))
dense_fixed   = load_dense_fixed(dense_loader_path=str(PATH_DENSE_LOADER))
graph_rag     = load_graph_rag(loader_path=str(PATH_GRAG_LOADER))
synthesizer   = AnswerSynthesizerAgent()

baseline_orchestrator = Orchestrator(
    bm25=bm25_fixed_qe, dense=dense_fixed,
    graph=graph_rag, synthesizer=synthesizer
)
print("✓ Baseline system ready.")

## Section 4 — Evidence Sufficiency Estimation [Mechanism A]

### Problem

The baseline passes retrieved documents directly to Mistral-7B without checking whether they contain relevant information. This is the root cause of hallucinations and overconfidence failures identified in the failure analysis.

### Solution: EvidenceSufficiencyChecker

Evaluates three independent signals on the retrieved documents **before** any answer is generated:

| Signal | Method | Threshold |
|---|---|---|
| `semantic_coverage` | Avg cosine similarity between query embedding and top-k doc embeddings (E5-large) | ≥ 0.35 |
| `chunk_support_count` | Number of docs with per-doc similarity ≥ 0.40 | ≥ 3 |
| `aspect_coverage_ratio` | Fraction of query key-terms found in combined retrieved text | ≥ 0.50 |
| `sufficiency_score` | Weighted composite (0.40/0.35/0.25) of the three above | ≥ 0.45 |

A **hard floor** of 0.20 bypasses recovery and triggers immediate abstention.

### Why three signals?

A single threshold is fragile: a document that paraphrases the answer without using exact query terms scores well on semantic similarity but poorly on aspect coverage — correctly diagnosing a `coverage_low` failure and triggering a targeted query rewrite.

> ⚠️ The `EvidenceSufficiencyChecker` reuses the `multilingual-E5-large` embeddings already loaded by the Dense retriever — no additional model is loaded.

In [ ]:
from rag.reliability.evidence_sufficiency import EvidenceSufficiencyChecker, EvidenceReport

# Reuse Dense retriever's embed function — no new model needed
checker = EvidenceSufficiencyChecker(embed_fn=dense_fixed.embed)
print("✓ EvidenceSufficiencyChecker ready.")

In [ ]:
def show_evidence_report(query: str, strategy: str = "confidence", top_k: int = 5):
    result = baseline_orchestrator.run(strategy=strategy, query=query, top_k=top_k)
    docs   = result["documents"]
    report = checker.check(query, docs)

    print(f"Query  : {query}")
    print(f"{'─'*60}")
    print(f"  semantic_coverage     : {report.semantic_coverage:.3f}   (threshold ≥ 0.35)")
    print(f"  chunk_support_count   : {report.chunk_support_count}       (threshold ≥ 3)")
    print(f"  aspect_coverage_ratio : {report.aspect_coverage_ratio:.3f}   (threshold ≥ 0.50)")
    print(f"{'─'*60}")
    print(f"  sufficiency_score     : {report.score:.3f}   → sufficient={report.sufficient}")
    print(f"  failure_type          : {report.failure_type}")
    if report.missing_aspects:
        print(f"  missing_aspects       : {report.missing_aspects}")
    return report

In [ ]:
print("=" * 62)
print("EXAMPLE 1 — likely sufficient evidence")
print("=" * 62)
_ = show_evidence_report("Who at ETH received ERC grants?")

print()
print("=" * 62)
print("EXAMPLE 2 — known baseline failure (hallucination case)")
print("=" * 62)
_ = show_evidence_report("Who was president of ETH in 2003?")

print()
print("=" * 62)
print("EXAMPLE 3 — ambiguous/broad query")
print("=" * 62)
_ = show_evidence_report("What research is ETH famous for?")

## Section 5 — Recovery Agent [Mechanism G — Adaptive Requirement]

### Why targeted recovery?

A generic "retry with the same query" explores no new evidence.  
The `RecoveryAgent` diagnoses which signal was weakest and selects a **different** corrective action on each attempt:

| `failure_type` | Recovery action | Rationale |
|---|---|---|
| `coverage_low` | LLM-based query rewrite | Query too narrow → rephrase to improve recall |
| `few_docs` | Strategy switch (Confidence → Voting → Waterfall) | Different fusion may surface complementary results |
| `low_similarity` | PRF expansion via QEBM25 | Vocabulary mismatch → expand with top-doc terms |
| `exhausted` | Signal AbstentionGate | Further retries are unlikely to yield better evidence |

This is the **adaptive requirement**: the system changes its orchestration behaviour dynamically based on observed evidence quality — not a fixed script.

Maximum **2 retries** are enforced to bound worst-case latency.

> ⚠️ LLM-based query rewriting requires an `OPENAI_API_KEY`.  
> Without it, a rule-based fallback appends missing key-terms to the original query.

In [ ]:
from rag.reliability.recovery import RecoveryAgent

# The RecoveryAgent can rewrite queries using an LLM (coverage_low failure type).
# This is optional — without a key the rule-based fallback appends missing key-terms instead.
#
# To use your OpenAI key securely in Colab without exposing it in the notebook:
#   1. Click the 🔑 "Secrets" icon in the left Colab sidebar
#   2. Add a secret named OPENAI_API_KEY with your key as the value
#   3. Toggle "Notebook access" ON
#
# The key is stored in YOUR Colab account only — teammates running this notebook
# from the shared Drive will not see it. They can add their own secret or leave it unset.

openai_client = None
try:
    from google.colab import userdata
    openai_api_key = userdata.get("OPENAI_API_KEY")
    if openai_api_key:
        from openai import OpenAI
        openai_client = OpenAI(api_key=openai_api_key)
        print("✓ OpenAI client ready — LLM-based query rewriting enabled.")
    else:
        print("ℹ  OPENAI_API_KEY secret not set — rule-based query rewrite fallback will be used.")
except Exception:
    print("ℹ  Could not read Colab secrets — rule-based query rewrite fallback will be used.")

recovery_agent = RecoveryAgent(max_retries=2, openai_client=openai_client)
print("✓ RecoveryAgent ready.")

In [ ]:
def show_recovery_actions(query: str, strategy: str = "confidence"):
    """Show what recovery actions would be selected for each attempt."""
    result = baseline_orchestrator.run(strategy=strategy, query=query, top_k=5)
    report = checker.check(query, result["documents"])

    print(f"Query        : {query}")
    print(f"sufficient   : {report.sufficient}  |  score={report.score:.3f}  |  failure='{report.failure_type}'")

    if report.sufficient:
        print("→ Evidence is sufficient — no recovery needed.")
        return

    current_strategy = strategy
    current_query    = query
    for attempt in range(recovery_agent.max_retries + 1):
        action = recovery_agent.choose_action(report, attempt=attempt, current_strategy=current_strategy)
        print(f"  {action.trace_entry}")
        if action.type == "abstain":
            break
        if action.type == "rewrite_query":
            current_query = recovery_agent.rewrite_query(current_query, report.missing_aspects)
            print(f"    → Rewritten query: '{current_query}'")
        elif action.type == "switch_strategy":
            current_strategy = action.new_strategy


print("=" * 62)
print("RECOVERY — few_docs failure")
print("=" * 62)
show_recovery_actions("Who was president of ETH in 2003?")

print()
print("=" * 62)
print("RECOVERY — coverage_low failure")
print("=" * 62)
show_recovery_actions("What are ETH sustainability initiatives for 2030?")

## Section 6 — Abstention Gate [Mechanism E]

### Structured abstention vs. silent failure

The baseline returns "NOT FOUND IN CONTEXT" or a hallucinated answer — both are opaque.  
The `AbstentionGate` produces a **structured `AbstentionResponse`** with:
- `trigger`: `"evidence_failure"` or `"trust_failure"`
- `reason`: human-readable explanation
- `missing_aspects`: which query terms were absent from the evidence
- `confidence`: always 0.0
- `trace`: full decision log

### Two entry paths

The AbstentionGate is the **unified terminal failure state**, reached from:
1. `abstain_evidence()` — retrieval failure (A hard floor, or G exhausted)
2. `abstain_low_trust()` — synthesis failure (H detects low trust) ← *teammate integrates here*

In [ ]:
from rag.reliability.abstention import AbstentionMechanism, AbstentionResponse

abstainer = AbstentionMechanism()
print("✓ AbstentionMechanism ready.")

In [ ]:
def show_abstention_evidence(query: str):
    """Demonstrate evidence_failure abstention."""
    result = baseline_orchestrator.run(strategy="confidence", query=query, top_k=5)
    report = checker.check(query, result["documents"])

    trace = [
        f"EvidenceCheck: score={report.score:.3f}, failure='{report.failure_type}'",
        "Recovery attempts exhausted."
    ]
    abstention = abstainer.abstain_evidence(query, report, trace)

    print(f"Query      : {query}")
    print(f"Trigger    : {abstention.trigger}")
    print(f"Reason     : {abstention.reason}")
    print(f"Missing    : {abstention.missing_aspects}")
    print(f"Confidence : {abstention.confidence}")
    print("Trace:")
    for e in abstention.trace:
        print(f"  {e}")


print("=" * 62)
print("ABSTENTION — evidence_failure path")
print("=" * 62)
show_abstention_evidence("Who was president of ETH in 2003?")

print()
print("=" * 62)
print("ABSTENTION — trust_failure path (triggered by H)")
print("=" * 62)
# ─────────────────────────────────────────────────────────────
# Introduce code here for H (TrustScorer)
# Once H is implemented, pass the real trust_score and groundedness_score.
# For now, a simulated call shows the structure of the response.
# ─────────────────────────────────────────────────────────────
abstention_trust = abstainer.abstain_low_trust(
    query="Who was president of ETH in 2003?",
    trust_score=0.0,          # placeholder — replace with H output
    groundedness_score=0.0,   # placeholder — replace with B output
    trace=["[H placeholder] trust_failure triggered"]
)
print(f"Trigger    : {abstention_trust.trigger}")
print(f"Reason     : {abstention_trust.reason}")
print(f"Trust      : {abstention_trust.trust_score}")

## Section 7 — Reliable Orchestrator (A + G + E)

The `ReliableOrchestrator` is the policy controller that wires mechanisms A, G, and E together.  
It wraps the existing `Orchestrator` and adds the evidence-based loop before synthesis.

B and H are injected as optional parameters. They are left as `None` here until teammates implement them.

In [ ]:
from rag.reliability.reliable_orchestrator import ReliableOrchestrator, ReliableResponse

# ─────────────────────────────────────────────────────────────
# Introduce code here for B (GroundednessVerifier)
# Replace None with an instance of B once implemented.
# ─────────────────────────────────────────────────────────────
groundedness_verifier = None

# ─────────────────────────────────────────────────────────────
# Introduce code here for H (TrustScorer)
# Replace None with an instance of H once implemented.
# ─────────────────────────────────────────────────────────────
trust_scorer = None

reliable_orchestrator = ReliableOrchestrator(
    orchestrator=baseline_orchestrator,
    embed_fn=dense_fixed.embed,
    max_retries=2,
    openai_client=openai_client,
    groundedness_verifier=groundedness_verifier,
    trust_scorer=trust_scorer,
)
print("✓ ReliableOrchestrator ready.")

In [ ]:
def run_and_show(query: str, strategy: str = "confidence"):
    t0       = time.time()
    response = reliable_orchestrator.run(query=query, strategy=strategy, top_k=5)
    elapsed  = time.time() - t0

    print(f"Query      : {query}")
    print(f"Abstained  : {response.abstained}  |  Recoveries: {response.recovery_attempts}  |  {elapsed:.2f}s")

    if response.abstained:
        print(f"Trigger    : {response.abstention.trigger}")
        print(f"Reason     : {response.abstention.reason}")
    else:
        print(f"Answer     : {response.answer}")
        if response.evidence_report:
            print(f"Evidence   : score={response.evidence_report.score:.3f}")

    print("Last trace entries:")
    for entry in response.trace[-5:]:
        print(f"  {entry}")
    print()

In [ ]:
print("=" * 62)
print("TEST 1 — Known failure: hallucination case")
print("=" * 62)
run_and_show("Who was president of ETH in 2003?")

print("=" * 62)
print("TEST 2 — Likely answerable")
print("=" * 62)
run_and_show("Who at ETH received ERC grants?")

print("=" * 62)
print("TEST 3 — Known failure: rectors query")
print("=" * 62)
run_and_show("Who were the rectors of ETH between 2017 and 2022?")

print("=" * 62)
print("TEST 4 — Cross-lingual (German)")
print("=" * 62)
run_and_show("Wer war 2003 Präsident der ETH?")

## Section 8 — Benchmark Run (A, G, E)

Run the reliable orchestrator over all 50 benchmark queries and report:
- Abstention rate and trigger breakdown
- Recovery statistics
- Comparison of answer rate vs baseline

In [ ]:
with open(PATH_QA, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

questions, gold_answers = [], []
for x in qa_data:
    questions.append(x["question"])
    gold_answers.append(x["answer"])
    questions.append(x["question_de"])
    gold_answers.append(x["answer_de"])

print(f"Loaded {len(questions)} queries.")

In [ ]:
reliable_results = []

for q, a in tqdm(zip(questions, gold_answers), total=len(questions), desc="Reliable pipeline"):
    t0 = time.time()
    r  = reliable_orchestrator.run(query=q, strategy="confidence", top_k=5)
    reliable_results.append({
        "query":      q,
        "gold":       a,
        "answer":     r.answer,
        "abstained":  r.abstained,
        "trigger":    r.abstention.trigger if r.abstained else None,
        "recoveries": r.recovery_attempts,
        "ev_score":   r.evidence_report.score if r.evidence_report else None,
        "latency":    time.time() - t0,
    })

rel_df = pd.DataFrame(reliable_results)
print("Done.")

In [ ]:
n = len(reliable_results)
abstained_df  = rel_df[rel_df["abstained"]]
recovered_df  = rel_df[rel_df["recoveries"] > 0]
answered_df   = rel_df[~rel_df["abstained"]]

print(f"Total queries    : {n}")
print(f"Answered         : {len(answered_df)}  ({len(answered_df)/n:.1%})")
print(f"Abstained        : {len(abstained_df)}  ({len(abstained_df)/n:.1%})")
print(f"Had recovery     : {len(recovered_df)}  ({len(recovered_df)/n:.1%})")
print(f"Avg latency      : {rel_df['latency'].mean():.2f}s")
print(f"Avg ev_score     : {rel_df['ev_score'].mean():.3f}")
print()

if len(abstained_df) > 0:
    print("Abstention trigger breakdown:")
    print(abstained_df["trigger"].value_counts().to_string())
    print()
    print("Abstained queries:")
    for _, row in abstained_df.iterrows():
        print(f"  [{row['trigger']:20s}] ev={row['ev_score']:.2f if row['ev_score'] else 'N/A'}  {row['query'][:65]}")

In [ ]:
if len(recovered_df) > 0:
    answered_after = recovered_df[~recovered_df["abstained"]]
    abstained_after = recovered_df[recovered_df["abstained"]]

    print(f"Recovery outcomes:")
    print(f"  Answered after recovery  : {len(answered_after)}")
    print(f"  Abstained after recovery : {len(abstained_after)}")
    print()
    print("Queries where recovery led to an answer:")
    for _, row in answered_after.iterrows():
        print(f"  [rec={int(row['recoveries'])}] ev={row['ev_score']:.2f}  {row['query'][:65]}")